# NYT Stock Headlines Sentiment Pipeline

1990~2025년 NYT "stock" 관련 기사 헤드라인 수집 → FinBERT 센티먼트 스코어링 → 분기별 집계

- **수집**: NYT Article Search API, `sort=relevance`, 월당 50건 (관련도 상위)
- **스코어링**: ProsusAI/FinBERT (GPU 활용)
- **출력**: `quarterly_sentiment.csv` → 항상성 RL 환경 observation에 병합

## 0. Setup

In [ ]:
!pip install -q requests transformers torch pandas

In [ ]:
import requests
import time
import csv
import os
import json
import calendar
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import drive

# Google Drive mount (checkpoint & CSV save)
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/homeostatic-market/news'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Save directory: {SAVE_DIR}')

## 1. NYT API Key

In [ ]:
# NYT Developer (https://developer.nytimes.com) 에서 발급받은 API Key 입력
API_KEY = 'V21sbXOFUnMAoigNw0YRlCrxahLKbKzToiamQpkEoECGMpiX'

## 2. Crawl Headlines

- 키워드: `"stock"`
- 정렬: `relevance` (월 전체에서 관련도 상위 50건 추출, 시간편향 없음)
- 월당 5페이지 (50건), 420개월 = 약 21,000건
- Rate Limit: 12초 간격, 429 시 65초 대기
- 체크포인트: 월+페이지 단위, 중단 후 셀 재실행하면 이어서 수집

In [ ]:
# ── Settings ──
BASE_URL = 'https://api.nytimes.com/svc/search/v2/articlesearch.json'
QUERY = 'stock'
START_YEAR = 1990
END_YEAR = 2025

REQUEST_DELAY = 12       # seconds between requests
RATE_LIMIT_WAIT = 65     # seconds on 429
EMPTY_RESULT_WAIT = 30   # seconds on empty result (hidden rate limit)
MAX_RETRIES = 3          # retries per page
MAX_PAGES = 5            # pages per month (5 pages = 50 articles)

CSV_FILE = os.path.join(SAVE_DIR, 'nyt_headlines.csv')
CHECKPOINT_FILE = os.path.join(SAVE_DIR, 'crawl_checkpoint.json')

CSV_COLUMNS = [
    'date', 'year_month', 'headline', 'abstract', 'lead_paragraph',
    'word_count', 'section', 'news_desk', 'type_of_material', 'keywords', 'web_url',
]


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return {'completed_months': [], 'current_month': None, 'current_page': 0}
    with open(CHECKPOINT_FILE, 'r') as f:
        return json.load(f)


def save_checkpoint(cp):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(cp, f, indent=2)


def init_csv():
    header_line = ','.join(CSV_COLUMNS)
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, 'w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
            writer.writeheader()
    else:
        with open(CSV_FILE, 'r', encoding='utf-8') as f:
            first_line = f.readline().strip()
        if first_line != header_line:
            with open(CSV_FILE, 'r', encoding='utf-8') as f:
                content = f.read()
            with open(CSV_FILE, 'w', encoding='utf-8', newline='') as f:
                f.write(header_line + '\n' + content)


def append_rows(rows):
    with open(CSV_FILE, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writerows(rows)


def extract_keywords(doc):
    return '; '.join(kw.get('value', '') for kw in doc.get('keywords', []) if kw.get('value'))


def parse_doc(doc, year_month):
    return {
        'date': doc.get('pub_date', ''),
        'year_month': year_month,
        'headline': doc.get('headline', {}).get('main', ''),
        'abstract': doc.get('abstract', ''),
        'lead_paragraph': doc.get('lead_paragraph', ''),
        'word_count': doc.get('word_count', 0),
        'section': doc.get('section_name', ''),
        'news_desk': doc.get('news_desk', ''),
        'type_of_material': doc.get('type_of_material', ''),
        'keywords': extract_keywords(doc),
        'web_url': doc.get('web_url', ''),
    }


def fetch_page(begin_date, end_date, page):
    params = {
        'q': QUERY,
        'begin_date': begin_date,
        'end_date': end_date,
        'api-key': API_KEY,
        'page': page,
        'sort': 'relevance',
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    if resp.status_code == 429:
        return 'rate_limit'
    if resp.status_code == 401:
        raise RuntimeError('401 Unauthorized - check API key')
    resp.raise_for_status()
    docs = resp.json().get('response', {}).get('docs')
    return docs if docs is not None else []


def fetch_month(year, month, start_page, checkpoint):
    _, last_day = calendar.monthrange(year, month)
    begin = f'{year}{month:02d}01'
    end = f'{year}{month:02d}{last_day:02d}'
    ym = f'{year}-{month:02d}'

    page = start_page
    total = 0
    consecutive_empty = 0

    while page < MAX_PAGES:
        retries = 0
        docs = None
        while retries < MAX_RETRIES:
            try:
                result = fetch_page(begin, end, page)
                if result == 'rate_limit':
                    print(f'    [429] page {page} - {RATE_LIMIT_WAIT}s wait...')
                    time.sleep(RATE_LIMIT_WAIT)
                    retries += 1
                    continue
                docs = result
                break
            except Exception as e:
                print(f'    [error] page {page}: {e}')
                time.sleep(30)
                retries += 1

        if docs is None:
            page += 1
            continue

        if not docs:
            consecutive_empty += 1
            if consecutive_empty == 1:
                time.sleep(EMPTY_RESULT_WAIT)
                continue
            elif consecutive_empty >= 3:
                break
            else:
                page += 1
                time.sleep(REQUEST_DELAY)
                continue

        consecutive_empty = 0
        rows = [parse_doc(d, ym) for d in docs]
        append_rows(rows)
        total += len(rows)
        checkpoint['current_page'] = page + 1
        save_checkpoint(checkpoint)
        page += 1
        time.sleep(REQUEST_DELAY)

    return total


print('Crawler ready.')

In [ ]:
# ── Run Crawler ──
# Re-run this cell to resume from checkpoint

init_csv()
checkpoint = load_checkpoint()
completed = set(checkpoint['completed_months'])
print(f'Completed months: {len(completed)}')
print(f'Range: {START_YEAR}-01 ~ {END_YEAR}-12')
print()

now = datetime.now()
grand_total = 0

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        if datetime(year, month, 1) > now:
            break

        ym = f'{year}-{month:02d}'
        if ym in completed:
            continue

        start_page = 0
        if checkpoint.get('current_month') == ym:
            start_page = checkpoint.get('current_page', 0)

        print(f'[{ym}] page {start_page}...', end=' ')

        checkpoint['current_month'] = ym
        checkpoint['current_page'] = start_page
        save_checkpoint(checkpoint)

        count = fetch_month(year, month, start_page, checkpoint)
        grand_total += count
        print(f'{count} articles (total: {grand_total})')

        checkpoint['completed_months'].append(ym)
        checkpoint['current_month'] = None
        checkpoint['current_page'] = 0
        save_checkpoint(checkpoint)
        completed.add(ym)

print(f'\nDone! {grand_total} new articles. File: {CSV_FILE}')

## 3. Check Collected Data

In [ ]:
df = pd.read_csv(CSV_FILE)
print(f'Total articles: {len(df)}')
print(f'Date range: {df["date"].min()} ~ {df["date"].max()}')
print(f'\nPer year_month (first 10):')
print(df['year_month'].value_counts().sort_index().head(10))
print(f'\nSample headlines:')
df[['date', 'headline']].head(10)

## 4. FinBERT Sentiment Scoring

- headline + abstract 결합 → FinBERT
- 출력: positive, negative, neutral 확률 + sentiment_score (positive - negative)
- Colab GPU (T4) 기준 약 2만건 = 5~10분

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(device)
model.eval()
print('FinBERT loaded.')

In [ ]:
SCORED_CSV = os.path.join(SAVE_DIR, 'nyt_headlines_scored.csv')
BATCH_SIZE = 64

df = pd.read_csv(CSV_FILE)
df = df[df['headline'].notna() & (df['headline'].str.strip() != '')].copy()
print(f'Scoring {len(df)} headlines...')

# headline + abstract
texts = []
for _, row in df.iterrows():
    t = str(row['headline'])
    a = str(row.get('abstract', ''))
    if a and a != 'nan':
        t = t + '. ' + a
    texts.append(t)

# Batch inference
all_scores = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1).cpu().numpy()
    for p in probs:
        all_scores.append({
            'positive': float(p[0]),
            'negative': float(p[1]),
            'neutral': float(p[2]),
            'sentiment_score': float(p[0] - p[1]),
        })
    if (i // BATCH_SIZE) % 50 == 0:
        print(f'  {min(i+BATCH_SIZE, len(texts))}/{len(texts)}')

scores_df = pd.DataFrame(all_scores)
df_scored = pd.concat([df.reset_index(drop=True), scores_df], axis=1)
df_scored.to_csv(SCORED_CSV, index=False)
print(f'\nScoring done. Saved to: {SCORED_CSV}')
print(f'Sentiment stats:')
print(df_scored['sentiment_score'].describe())

## 5. Quarterly Aggregation

In [ ]:
QUARTERLY_CSV = os.path.join(SAVE_DIR, 'quarterly_sentiment.csv')

df_scored = pd.read_csv(SCORED_CSV)
df_scored['date'] = pd.to_datetime(df_scored['date'], errors='coerce')
df_scored = df_scored.dropna(subset=['date'])
df_scored['quarter'] = df_scored['date'].dt.to_period('Q')

quarterly = df_scored.groupby('quarter').agg(
    sentiment_mean=('sentiment_score', 'mean'),
    sentiment_std=('sentiment_score', 'std'),
    sentiment_median=('sentiment_score', 'median'),
    positive_ratio=('positive', 'mean'),
    negative_ratio=('negative', 'mean'),
    neutral_ratio=('neutral', 'mean'),
    article_count=('headline', 'count'),
).reset_index()

quarterly['quarter'] = quarterly['quarter'].astype(str)
quarterly.to_csv(QUARTERLY_CSV, index=False)

print(f'Quarterly sentiment saved to: {QUARTERLY_CSV}')
print(f'\n{len(quarterly)} quarters, {quarterly["article_count"].sum()} total articles')
quarterly.head(20)

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

x = range(len(quarterly))
labels = quarterly['quarter'].values

# Sentiment mean
axes[0].bar(x, quarterly['sentiment_mean'], color=['g' if v > 0 else 'r' for v in quarterly['sentiment_mean']], alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Sentiment (mean)')
axes[0].set_title('NYT Stock Headlines Sentiment (1990-2025, quarterly)')

# Positive / Negative ratio
axes[1].plot(x, quarterly['positive_ratio'], 'g-', label='Positive', alpha=0.7)
axes[1].plot(x, quarterly['negative_ratio'], 'r-', label='Negative', alpha=0.7)
axes[1].set_ylabel('Ratio')
axes[1].legend()

# Article count
axes[2].bar(x, quarterly['article_count'], color='steelblue', alpha=0.7)
axes[2].set_ylabel('Article count')

# X-axis labels (every 4 quarters = 1 year)
tick_pos = [i for i in x if i % 4 == 0]
tick_labels = [labels[i][:4] for i in tick_pos]
axes[2].set_xticks(tick_pos)
axes[2].set_xticklabels(tick_labels, rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sentiment_quarterly.png'), dpi=150)
plt.show()
print('Plot saved.')

## 7. Download quarterly_sentiment.csv

Google Drive에 자동 저장되어 있으므로 별도 다운로드 불필요.  
로컬로 직접 받으려면 아래 셀 실행.

In [ ]:
from google.colab import files
files.download(QUARTERLY_CSV)